# Aviation Girl V3 Training
Train Qwen2.5-7B (or 3B) with LoRA on Google Colab (Free T4 GPU)

**Model Options:**
- **7B**: Smarter responses, needs 4-bit quantization, may OOM on busy Colab instances
- **3B**: Faster, more reliable, proven to work on free T4

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q torch transformers peft bitsandbytes accelerate datasets

In [ ]:
# Verify GPU and check memory
import torch

if not torch.cuda.is_available():
    print("  NO GPU DETECTED!")
    print("Go to Runtime > Change runtime type > Select T4 GPU")
    raise RuntimeError("GPU required for training")

print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"  Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"  Available Memory: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3:.1f} GB")

# Clear any cached memory
torch.cuda.empty_cache()
print("  Memory cleared")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 2. Load Training Data

In [ ]:
# Load training data from Google Drive
from datasets import load_dataset

# Path to your training data in Google Drive
data_path = "/content/drive/MyDrive/aviation_girl/v3_training_data.jsonl"

# Alternative: If you put it in the root of Drive
# data_path = "/content/drive/MyDrive/v3_training_data.jsonl"

print(f"Loading data from {data_path}...")
dataset = load_dataset('json', data_files=data_path)
print(f"  Loaded {len(dataset['train'])} training examples")

# Preview first example
print("\nFirst example:")
print(dataset['train'][0])

In [ ]:
# Preprocess: Convert messages to model input format
def format_chat(example):
    # Apply chat template to convert messages to text
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    # Tokenize
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding=False,
    )
    # Labels are the same as input_ids for causal LM
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

print("Preprocessing dataset...")
tokenized_dataset = dataset['train'].map(
    format_chat,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing"
)
print(f"  Preprocessed {len(tokenized_dataset)} examples")

## 3. Load Model with 4-bit Quantization

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
import gc

# Clear memory before loading
gc.collect()
torch.cuda.empty_cache()

# OPTION 1: Qwen2.5-7B (bigger, smarter, but needs 4-bit quantization)
model_name = "Qwen/Qwen2.5-7B-Instruct"

# OPTION 2: Qwen2.5-3B (smaller, faster, more reliable on free Colab)
# model_name = "Qwen/Qwen2.5-3B-Instruct"

# OPTION 3: Qwen2.5-1.5B (smallest, guaranteed to work)
# model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading {model_name}...")
print(f"Available memory: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# For 7B: Use 4-bit quantization to fit in memory
if "7B" in model_name:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,  # Extra compression
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,  # Important for loading large models
    )
    model = prepare_model_for_kbit_training(model)
else:
    # For 3B/1.5B: Use float16 (simpler, more stable)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        use_cache=False,
        low_cpu_mem_usage=True,
    )
    model.gradient_checkpointing_enable()

print(f"  Model loaded on {model.device}")
print(f"Memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## 4. Configure LoRA

In [ ]:
print("Applying LoRA...")

# LoRA config - minimal rank to save memory on 7B
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,  # Lower rank for 7B to save memory (16 for 3B)
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],  # Only 2 modules for 7B (4 for 3B)
    bias="none"
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("  LoRA applied!")

## 5. Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Data collator for dynamic padding
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing causal LM, not masked LM
)

# Training arguments - aggressive memory optimization for 7B
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,  # Higher for 7B (4 for 3B)
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_steps=200,  # Less frequent saves for 7B
    save_total_limit=1,  # Keep only 1 checkpoint to save space
    bf16=True,  # bfloat16 for 7B (fp16 for 3B)
    optim="paged_adamw_8bit",  # Memory-efficient optimizer for 7B
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    report_to="none",
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,  # Use preprocessed dataset
    data_collator=data_collator,
)

# Train!
print("Starting training...")
print("⚠️ If OOM occurs, switch to 3B model (uncomment line in cell 3)")
trainer.train()
print("  Training complete!")

## 6. Save Adapter

In [ ]:
# Save to Drive
output_dir = "/content/drive/MyDrive/aviation_girl_v3_adapter"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"  Saved to {output_dir}")

## 7. Test

In [ ]:
# Test the model
test_prompt = "User: heya! what's your favorite plane?\nAssistant:"

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.8, do_sample=True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)